In [2]:
%%capture

import re

import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
%pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
%pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install --no-deps --upgrade "torchao>=0.16.0"
%pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
%pip install torchcodec

import torch; torch._dynamo.config.recompile_limit = 64;

%pip install --no-deps --upgrade timm  # For Gemma 4 vision/audio

%pip install evaluate datasets==3.6.0 rouge_score

In [3]:
from unsloth import FastModel

MODEL_NAME = 'unsloth/gemma-4-E2B-it'

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [17]:
from transformers import TextStreamer

def do_gemma_4_inference(messages, max_new_tokens = 128, with_streamer = True):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")
    
    if not with_streamer:
        return model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature = 1.0, top_p = 0.95, top_k = 64, # recommended by devs
            use_cache = True
        ), inputs
    
    _ = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64, # recommended by devs
        streamer = TextStreamer(tokenizer, skip_prompt = True),
        use_cache = True
    )

In [23]:
from pandas import DataFrame
from datasets import load_dataset, Dataset

dataset = load_dataset("webis/tldr-17", split="train", trust_remote_code=True, streaming=True)
dataset = dataset.take(50)

df = DataFrame(list(dataset))
dataset = Dataset.from_pandas(df)

In [28]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")
model.eval()

system_msg = (
    "You are a helpful assistant that writes concise TL;DR summaries "
    "of Reddit posts in one or two sentences."
)

predictions, references = [], []

eval_subset = dataset.select(range(3))

for example in tqdm(eval_subset, desc="Generating summaries"):
    content = example["content"]
    ref = example["summary"]
    
    messages = [
        {
            "role": "system",
            "content": [{
                "type" : "text",
                "text" : system_msg,
            }]
        },
        {
            "role": "user",
            "content": [{
                "type" : "text",
                "text" : f"SUBREDDIT:\n{example["subreddit"]}\nREDDIT POST:\n{example["content"]}",
            }]
        },
        {
            "role": "assistant",
            "content": [{
                "type" : "text",
                "text" : example["summary"],
            }]
        },
    ]
    
    outputs, inputs = do_gemma_4_inference(messages, with_streamer=False)

    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    
    print('>>> pred', pred)
    print('>>> ref', ref)
    
    predictions.append(pred)
    references.append(ref)

results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
)

for k, v in results.items():
    print(f"  {k}: {v:.4f}")

Generating summaries:  33%|███▎      | 1/3 [00:04<00:08,  4.03s/it]

>>> pred The author argues that moving to a standard time like UTC or UTC+1 would simplify timekeeping systems, as modern technology and efficient lighting make complex seasonal adjustments unnecessary.
>>> ref Shifting seasonal time is no longer worth it.


Generating summaries:  67%|██████▋   | 2/3 [00:07<00:03,  3.41s/it]

>>> pred Art is subjective, and the poster finds some street art styles repetitive, like CLET's use of recurring images.
>>> ref Personal opinions 'n shit.


Generating summaries: 100%|██████████| 3/3 [00:10<00:00,  3.36s/it]

>>> pred The poster finds the Wall Street Journal bland and full of unnecessary jargon, suggesting it targets a specific demographic with overly complicated language.
>>> ref insults and slack ass insight. 
 Wall Street Journal misses on enough counts that not only did i yawn with boredom, i fell asleep trying to read through this crap. 
 It may be the paper for you, but if your in the the market for a new read, i'd at least counsel you on reading the Washington Post instead, due out on stands anytime, or the Globe, which is slated for a weekly release.
  rouge1: 0.0979
  rouge2: 0.0140
  rougeL: 0.0524
  rougeLsum: 0.0730
